In [4]:
import pandas as pd
from odf import opendocument
from odf.table import Table, TableRow, TableCell
from odf.text import P

def read_ods(file_path):
    """Liest eine .ods-Datei und gibt ein pandas DataFrame zurück."""
    doc = opendocument.load(file_path)
    tables = doc.getElementsByType(Table)
    if not tables:
        raise ValueError("Keine Tabellen in der ODS-Datei gefunden")

    sheet = tables[0]
    rows = sheet.getElementsByType(TableRow)

    header = [str(cell) for cell in rows[0].getElementsByType(TableCell)]
    data = []
    for row in rows[1:]:
        cells = row.getElementsByType(TableCell)
        data.append([str(cell) for cell in cells])

    return pd.DataFrame(data, columns=header), doc, sheet

def vergleiche_eautos_ods(eingabe_pfad, ausgabe_pfad, fabia_verkaufspreis, benzinpreis=2.20, strompreis=0.30):
    """
    Vergleicht mehrere E-Autos mit Fabia und fügt Ergebnisse der ODS-Datei hinzu.

    Args:
        eingabe_pfad: Pfad zur Eingabe-ODS (Spalten: Name, Kaufpreis, Verbrauch)
        fabia_verkaufspreis: Verkaufspreis des Fabia in €
        benzinpreis: Benzinpreis in €/l (Default: 2.20)
        strompreis: Strompreis in €/kWh (Default: 0.30)
    """
    df, doc, sheet = read_ods(eingabe_pfad)

    fabia_verbrauch = 4.5  # l/100km
    fabia_kosten_pro_km = (fabia_verbrauch / 100) * benzinpreis

    # Neue Spaltenüberschriften
    new_headers = [
        'Fabia_Verkaufspreis', 'Benzinpreis', 'Strompreis',
        'Fabia_Kosten_pro_km', 'Eauto_Kosten_pro_km', 'Ersparnis_pro_km',
        'Preisunterschied', 'Amortisation_km', 'Amortisation_Jahre_10k',
        'Amortisation_Jahre_20k', 'Lohnt_sich'
    ]

    # Header-Reihe aktualisieren
    header_row = sheet.getElementsByType(TableRow)[0]
    for header in new_headers:
        new_cell = TableCell()
        new_cell.addElement(P(text=str(header)))
        header_row.addElement(new_cell)

    # Datenreihen aktualisieren
    data_rows = sheet.getElementsByType(TableRow)[1:]
    for i, (_, row) in enumerate(df.iterrows()):
        eauto_name = row['Name']
        eauto_preis = float(row['Kaufpreis'])
        eauto_verbrauch = float(row['Verbrauch'])

        eauto_kosten_pro_km = (eauto_verbrauch / 100) * strompreis
        preis_differenz = eauto_preis - fabia_verkaufspreis
        ersparnis_pro_km = fabia_kosten_pro_km - eauto_kosten_pro_km

        if ersparnis_pro_km <= 0:
            amortisation_km = "NIE"
            amortisation_jahre_10k = "NIE"
            amortisation_jahre_20k = "NIE"
        else:
            amortisation_km = preis_differenz / ersparnis_pro_km
            amortisation_jahre_10k = amortisation_km / 10000
            amortisation_jahre_20k = amortisation_km / 20000

        new_values = [
            fabia_verkaufspreis, benzinpreis, strompreis,
            fabia_kosten_pro_km, eauto_kosten_pro_km, ersparnis_pro_km,
            preis_differenz, amortisation_km, amortisation_jahre_10k,
            amortisation_jahre_20k, "JA" if ersparnis_pro_km > 0 else "NEIN"
        ]

        data_row = data_rows[i]
        for value in new_values:
            new_cell = TableCell()
            new_cell.addElement(P(text=str(value)))
            data_row.addElement(new_cell)

    # Dokument speichern
    doc.save(ausgabe_pfad)
    print(f"Ergebnisse wurden zur geschrieben!")
    return df

# Beispielaufruf
if __name__ == "__main__":
    FABIA_VERKAUFSPREIS = 14000
    EINGABE_PFAD = "E Autos.ods"
    AUSGABE_PFAD = "E Autos vergleich.ods"

    vergleiche_eautos_ods(
        eingabe_pfad=EINGABE_PFAD,
        ausgabe_pfad= AUSGABE_PFAD,
        fabia_verkaufspreis=FABIA_VERKAUFSPREIS,
        strompreis=0.30
    )

Ergebnisse wurden zur geschrieben!


In [ ]:
# eauto_vergleich.py
def vergleiche_eauto():
    print("Elektroauto vs. Skoda Fabia Vergleich")
    print("------------------------------------\n")

    # Dein Fabia (Benzin)
    fabia_verbrauch = 4.5  # l/100km
    benzinpreis = 2.20  # €/l (vorgegeben)
    fabia_verkaufspreis = float(input("Veraufspreis deines Fabia in €: "))

    # Strompreis
    strompreis = float(input("Aktueller Strompreis in €/kWh [0.30]: ") or 0.30)

    # Neues E-Auto
    print("\nDaten des Elektroautos:")
    eauto_name = input("Name des E-Autos: ")
    eauto_preis = float(input("Kaufpreis in €: "))
    eauto_verbrauch = float(input("Verbrauch in kWh/100km: "))

    # Berechnungen
    # Kosten pro km
    fabia_kosten_pro_km = (fabia_verbrauch / 100) * benzinpreis
    eauto_kosten_pro_km = (eauto_verbrauch / 100) * strompreis

    # Preisunterschied
    preis_differenz = eauto_preis - fabia_verkaufspreis

    # Kostenersparnis pro km
    ersparnis_pro_km = fabia_kosten_pro_km - eauto_kosten_pro_km

    if ersparnis_pro_km <= 0:
        print("\n❌ Das E-Auto ist pro km TEURER oder gleich teuer wie dein Fabia!")
        print("   -> Lohnt sich NIE rein über Energiekosten!")
        return

    # Amortisation
    amortisation_km = preis_differenz / ersparnis_pro_km

    # Ausgabe
    print("\n" + "="*60)
    print("ERGEBNIS:")
    print("="*60)
    print(f"\nDein Skoda Fabia (Benzin @ {benzinpreis}€/l):")
    print(f"  - Verbrauch: {fabia_verbrauch} l/100km")
    print(f"  - Energiekosten pro km: {fabia_kosten_pro_km:.3f} €")
    print(f"  - Verkaufserlös: {fabia_verkaufspreis:,.2f} €")

    print(f"\n{eauto_name} (Strom @ {strompreis}€/kWh):")
    print(f"  - Verbrauch: {eauto_verbrauch} kWh/100km")
    print(f"  - Energiekosten pro km: {eauto_kosten_pro_km:.3f} €")
    print(f"  - Kaufpreis: {eauto_preis:,.2f} €")
    print(f"  - Ersparnis pro km: {ersparnis_pro_km:.3f} €")

    print(f"\n💡 AMORTISATION:")
    print(f"   Das E-Auto lohnt sich nach **{amortisation_km:,.0f} km**!")

    print("\n📊 Praktische Beispiele (Jährliche Fahrleistung):")
    for km_pro_jahr in [10000, 15000, 20000, 25000, 30000]:
        jahre = amortisation_km / km_pro_jahr
        print(f"   - Bei {km_pro_jahr:>5,} km/Jahr: nach {jahre:>5.1f} Jahren")
        print(f"   - Bei {km_pro_jahr/12:>5,.0f} km/Monat: nach {jahre:>5.1f} Jahren")

def vergleiche_mehrere_eautos():
    print("\nMehrere E-Autos mit Fabia vergleichen")
    print("-----------------------------------\n")

    fabia_verbrauch = 4.5
    benzinpreis = 2.20
    fabia_verkaufspreis = float(input("Veraufspreis deines Fabia in €: "))
    strompreis = float(input("Aktueller Strompreis in €/kWh [0.30]: ") or 0.30)
    anzahl_autos = int(input("Wie viele E-Autos zum Vergleichen? "))

    autos = []
    for i in range(anzahl_autos):
        print(f"\nE-Auto {i+1}:")
        name = input("  Name: ")
        preis = float(input("  Kaufpreis in €: "))
        verbrauch = float(input("  Verbrauch in kWh/100km: "))
        autos.append((name, preis, verbrauch))

    # Fabia Kosten pro km
    fabia_kosten_pro_km = (fabia_verbrauch / 100) * benzinpreis

    print("\n" + "="*75)
    print(f"{'E-Auto':<20} {'Preis':>12} {'Verbrauch':>12} {'Kosten/km':>12} {'Amortisation':>15}")
    print("="*75)

    for name, preis, verbrauch in sorted(autos, key=lambda x: x[2]):  # Sortiert nach Verbrauch
        eauto_kosten_pro_km = (verbrauch / 100) * strompreis
        preis_diff = preis - fabia_verkaufspreis
        ersparnis_pro_km = fabia_kosten_pro_km - eauto_kosten_pro_km

        if ersparnis_pro_km <= 0:
            amort_km = "NIE"
        else:
            amort_km = f"{preis_diff / ersparnis_pro_km:,.0f} km"

        print(f"{name:<20} {preis:>12,.0f} € {verbrauch:>11.1f} kWh/100km {eauto_kosten_pro_km:>11.3f} € {amort_km:>15}")

    print("\n💡 Tipp: Je niedriger die Amortisations-KM, desto schneller lohnt sich das E-Auto!")

def main():
    while True:
        print("\n" + "="*60)
        print("ELEKTROAUTO-VERGLEICH MENÜ:")
        print("="*60)
        print("1. Ein E-Auto mit Fabia vergleichen")
        print("2. Mehrere E-Autos mit Fabia vergleichen")
        print("3. Beenden")
        choice = input("\nWahl [1-3]: ")

        if choice == "1":
            vergleiche_eauto()
        elif choice == "2":
            vergleiche_mehrere_eautos()
        elif choice == "3":
            print("\nAuf Wiedersehen!")
            break
        else:
            print("Ungültige Eingabe!")

if __name__ == "__main__":
    main()

In [8]:
import pandas as pd
from odf import opendocument
from odf.table import Table, TableRow, TableCell


def read_ods(file_path):
    """Liest eine .ods-Datei und gibt ein pandas DataFrame zurück."""
    doc = opendocument.load(file_path)
    # Get all tables from the document
    tables = doc.getElementsByType(Table)
    if not tables:
        raise ValueError("Keine Tabellen in der ODS-Datei gefunden")
    
    sheet = tables[0]  # Nimmt das erste Tabellenblatt
    rows = sheet.getElementsByType(TableRow)
    
    # Extrahiere Header und Daten
    header = [str(cell) for cell in rows[0].getElementsByType(TableCell)]
    data = []
    for row in rows[1:]:
        cells = row.getElementsByType(TableCell)
        data.append([str(cell) for cell in cells])
    
    import pandas as pd
    return pd.DataFrame(data, columns=header)
# Dann in deiner Funktion:
def vergleiche_eautos_excel(eingabe_pfad, ausgabe_pfad, fabia_verkaufspreis, benzinpreis=2.20, strompreis=0.30):
    # Eingabe lesen (jetzt für .ods)
    df = read_ods(eingabe_pfad)
    # Rest deines Codes bleibt gleich...
    """
    Vergleicht mehrere E-Autos mit Fabia basierend auf ODS-Daten.

    Args:
        eingabe_pfad: Pfad zur Eingabe-Excel (Spalten: Name, Kaufpreis, Verbrauch)
        ausgabe_pfad: Pfad zur Ausgabe-Excel
        fabia_verkaufspreis: Verkaufspreis des Fabia in €
        benzinpreis: Benzinpreis in €/l (Default: 2.20)
        strompreis: Strompreis in €/kWh (Default: 0.30)
    """
    # Eingabe lesen

    # Fabia-Daten
    fabia_verbrauch = 4.5  # l/100km
    fabia_kosten_pro_km = (fabia_verbrauch / 100) * benzinpreis

    # Berechnungen für jedes E-Auto
    results = []
    for _, row in df.iterrows():
        eauto_name = row['Name']
        eauto_preis = float(row['Kaufpreis'])
        eauto_verbrauch = float(row['Verbrauch'])  # kWh/100km

        # Kosten pro km
        eauto_kosten_pro_km = (eauto_verbrauch / 100) * strompreis

        # Preisunterschied
        preis_differenz = eauto_preis - fabia_verkaufspreis

        # Ersparnis pro km
        ersparnis_pro_km = fabia_kosten_pro_km - eauto_kosten_pro_km

        # Amortisation
        if ersparnis_pro_km <= 0:
            amortisation_km = "NIE"
            amortisation_jahre_10k = "NIE"
            amortisation_jahre_20k = "NIE"
        else:
            amortisation_km = preis_differenz / ersparnis_pro_km
            amortisation_jahre_10k = amortisation_km / 10000
            amortisation_jahre_20k = amortisation_km / 20000

        results.append({
            **row.to_dict(),
            'Fabia_Verkaufspreis': fabia_verkaufspreis,
            'Benzinpreis': benzinpreis,
            'Strompreis': strompreis,
            'Fabia_Kosten_pro_km': fabia_kosten_pro_km,
            'Eauto_Kosten_pro_km': eauto_kosten_pro_km,
            'Ersparnis_pro_km': ersparnis_pro_km,
            'Preisunterschied': preis_differenz,
            'Amortisation_km': amortisation_km,
            'Amortisation_Jahre_10k': amortisation_jahre_10k,
            'Amortisation_Jahre_20k': amortisation_jahre_20k,
            'Lohnt_sich': "JA" if ersparnis_pro_km > 0 else "NEIN"
        })

    # DataFrame für Ausgabe erstellen
    ergebnis_df = pd.DataFrame(results)

    # In Excel speichern
    ergebnis_df.to_excel(ausgabe_pfad, index=False)

    print(f"✅ Ergebnisse wurden in {ausgabe_pfad} gespeichert!")
    return ergebnis_df

# Beispielaufruf
if __name__ == "__main__":
    # Parameter (können auch aus einer Config-Datei oder CLI-Args kommen)
    FABIA_VERKAUFSPREIS = 14000  # Beispielwert
    EINGABE_PFAD = "E Autos.ods"
    AUSGABE_PFAD = "eauto_ergebnisse.xlsx"

    # Ausführen
    vergleiche_eautos_excel(
        eingabe_pfad=EINGABE_PFAD,
        ausgabe_pfad=AUSGABE_PFAD,
        fabia_verkaufspreis=FABIA_VERKAUFSPREIS,
        strompreis=0.30
    )

✅ Ergebnisse wurden in eauto_ergebnisse.xlsx gespeichert!
